# Neural network for AAMI classification

In this notebook we are gonna develop a fully connected neural network for classifying an ECG into an arrythmia class. First, the model is trained for classifying the ECG into the 5 AAMI classes. Then, we are going to repeat the process for a full 15 class classification.

## Imports

First, let's import the packages that we need. We also need to load in the environment the functions written in ../../src/data_preprocessing.py.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.neural_network import MLPClassifier
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_preprocessing.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or a notebook subfolder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_preprocessing import (
    group_aami_classes,
    standardize_data,
    standardize_and_pca,
    raw_pca,
)

Now we import the dataset and apply the mapping in the five classes. For the moment, we don't apply any kind of pre-processing.

In [ ]:
from pathlib import Path

data_dir = PROJECT_ROOT / "data"
if not all((data_dir / name).is_file() for name in (
    "train_data.csv", "val_data.csv", "test_data.csv"
)):
    raise FileNotFoundError("Dataset missing: see data/README.md for the required CSV files.")


train_data = pd.read_csv(data_dir / "train_data.csv", header=None)
val_data = pd.read_csv(data_dir / "val_data.csv", header=None)
test_data = pd.read_csv(data_dir / "test_data.csv", header=None)

print("Training set shape:", train_data.shape)
print("Validation set shape:", val_data.shape)
print("Test set shape:", test_data.shape)

train_data.head()

Let's split features and labels and apply the mapping. 

In [ ]:
# Separate features and labels
X_train = train_data.iloc[:, :-1].to_numpy()
y_train = train_data.iloc[:, -1].astype(int).to_numpy()

X_val = val_data.iloc[:, :-1].to_numpy()
y_val = val_data.iloc[:, -1].astype(int).to_numpy()

X_test = test_data.iloc[:, :-1].to_numpy()
y_test = test_data.iloc[:, -1].astype(int).to_numpy()

# Group the original labels into the five AAMI classes
y_train = group_aami_classes(y_train)
y_val = group_aami_classes(y_val)
y_test = group_aami_classes(y_test)

## Neural networks on raw data

Now we are ready for the training of the neural network. We will start with a simple two-layers network and then we will apply some fixed split validation to increase the generalization performance.

We compare architectures, regularization and preprocessing choices
using macro F1 on the supplied fixed validation set. Models are
fitted on the training set, and the test set is reserved for the
final evaluation.

In [ ]:
mlp_raw = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw.fit(X_train, y_train)

print("Epochs:", mlp_raw.n_iter_)
print("Final training loss:", mlp_raw.loss_)

The training stopped after 156 epochs. Let's see the performance on the validation set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = mlp_raw.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

This is already pretty good. Let's save these results and change some parameters of the network. In the end we will compare all the parameters. 
After changing the parameters we will repeat the process for standardized dataset and standardized+pca dataset.

Let's try with a network with only one layer with 100 neurons.

In [ ]:
mlp_raw_1layer = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw_1layer.fit(X_train, y_train)

print("Epochs:", mlp_raw_1layer.n_iter_)
print("Final training loss:", mlp_raw_1layer.loss_)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = mlp_raw_1layer.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

Let's now try with a more deep network, and then with one that penalizes more bigger weights (bigger alpha).

In [ ]:
mlp_raw_3layer = MLPClassifier(
    hidden_layer_sizes=(100,100,100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw_3layer.fit(X_train, y_train)

print("Epochs:", mlp_raw_3layer.n_iter_)
print("Final training loss:", mlp_raw_3layer.loss_)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = mlp_raw_3layer.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

In [ ]:
mlp_raw_alpha = MLPClassifier(
    hidden_layer_sizes=(100,100),
    activation="relu",
    solver="adam",
    alpha=0.001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw_alpha.fit(X_train, y_train)

print("Epochs:", mlp_raw_alpha.n_iter_)
print("Final training loss:", mlp_raw_alpha.loss_)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = mlp_raw_alpha.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

Let's try to change the number of neurons for the network with the best performance.

In [ ]:
mlp_raw_50 = MLPClassifier(
    hidden_layer_sizes=(50, 50),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw_50.fit(X_train, y_train)

print("Epochs:", mlp_raw_50.n_iter_)
print("Final training loss:", mlp_raw_50.loss_)

In [ ]:
y_val_pred = mlp_raw_50.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

In [ ]:
mlp_raw_200 = MLPClassifier(
    hidden_layer_sizes=(200, 200),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_raw_200.fit(X_train, y_train)

print("Epochs:", mlp_raw_200.n_iter_)
print("Final training loss:", mlp_raw_200.loss_)

In [ ]:
y_val_pred = mlp_raw_200.predict(X_val)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1, 2, 3, 4]))

## Neural network on standardized data

Now that the basic training and validation on raw data has been done, we try to see how performance change if we repeat the process for standardized data.
For the sake of comparison, we are going to start from the best network architecture found in the last validation. 

In [ ]:
X_train_scaled, X_val_scaled, X_test_scaled = standardize_data(
    X_train, X_val, X_test
)

In [ ]:
mlp_scaled = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_scaled.fit(X_train_scaled, y_train)

print("Epochs:", mlp_scaled.n_iter_)
print("Final training loss:", mlp_scaled.loss_)

In [ ]:
y_val_pred = mlp_scaled.predict(X_val_scaled)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

Let's try with more neurons.

In [ ]:
mlp_scaled_200 = MLPClassifier(
    hidden_layer_sizes=(200, 200),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_scaled_200.fit(X_train_scaled, y_train)

print("Epochs:", mlp_scaled_200.n_iter_)
print("Final training loss:", mlp_scaled_200.loss_)

In [ ]:
y_val_pred = mlp_scaled_200.predict(X_val_scaled)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

It seems that we are not able to do better than what we obtained with raw data with this standardized dataset. 
Let's move then to standardized+pca data.

## NN on standardized+PCA data

Let's now repeat the process. Let's keep the 95% of the variance of data.

In [ ]:
X_train_pca, X_val_pca, X_test_pca = standardize_and_pca(
    X_train, 0.95, X_val, X_test
)

print("Original number of features:", X_train.shape[1])
print("Number of PCA components:", X_train_pca.shape[1])
print("Training set shape:", X_train_pca.shape)
print("Validation set shape:", X_val_pca.shape)

Again, let's start with a simple network with $\alpha=0.0001$ and two layers with 100 neurons each.

In [ ]:
mlp_pca = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_pca.fit(X_train_pca, y_train)

print("Epochs:", mlp_pca.n_iter_)
print("Final training loss:", mlp_pca.loss_)

In [ ]:
y_val_pred = mlp_pca.predict(X_val_pca)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

Since with only standardization we did not improve the performance of the network on the validation set, it is reasonable to apply PCA directly on raw data. Let's keep 99% of the variance.

In [ ]:
import importlib
import data_preprocessing

importlib.reload(data_preprocessing)

from data_preprocessing import (
    group_aami_classes,
    standardize_data,
    standardize_and_pca,
    raw_pca,
)

In [ ]:
X_train_pca_raw, X_val_pca_raw, X_test_pca_raw = raw_pca(
    X_train, 0.99, X_val, X_test
)

In [ ]:
mlp_pca_raw = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_pca_raw.fit(X_train_pca_raw, y_train)

print("Epochs:", mlp_pca_raw.n_iter_)
print("Final training loss:", mlp_pca_raw.loss_)

In [ ]:
y_val_pred = mlp_pca_raw.predict(X_val_pca_raw)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

## Neural network on the spectrum

Assuming that the best architecture for the neural network is the one with two layers, 100 neurons each. An interesting experiment that we can do is to apply fft to see if the frequency spectrum of signals is a good feature space for a ML model. In the first place the spectrum is kept divided into imaginary and real parts, obtaining 252 features. This setup is tested only to see if the network is more capable of classifying based on the spectrum rather than on the amplitude, not to reduce the dimension of the problem.

In this case applying standardization might be a good pre-processing procedure as the fft can produce features with different scales.

In [ ]:
import importlib
import data_preprocessing

importlib.reload(data_preprocessing)
from data_preprocessing import fft_data

X_train_fft, X_val_fft, X_test_fft = fft_data(
    X_train, X_val, X_test
)

print("Training set shape:", X_train_fft.shape)
print("Validation set shape:", X_val_fft.shape)

In [ ]:
mlp_fft = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_fft.fit(X_train_fft, y_train)

print("Epochs:", mlp_fft.n_iter_)
print("Final training loss:", mlp_fft.loss_)

In [ ]:
y_val_pred = mlp_fft.predict(X_val_fft)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

This was on raw data. Let's try with standardize data.

In [ ]:
X_train_fft_scaled, X_val_fft_scaled, X_test_fft_scaled = standardize_data(
    X_train_fft, X_val_fft, X_test_fft
)

In [ ]:
mlp_fft_scaled = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_fft_scaled.fit(X_train_fft_scaled, y_train)

print("Epochs:", mlp_fft_scaled.n_iter_)
print("Final training loss:", mlp_fft_scaled.loss_)

In [ ]:
y_val_pred = mlp_fft_scaled.predict(X_val_fft_scaled)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

Until now the NN model trained on the fft data was trained using the complex spectrum, with 252 features. Let's try to see the performance when we use the magnitude of the spectrum. This solution uses only 126 features, but it might have worse performance.

In [ ]:
X_train_fft_mag = np.abs(np.fft.rfft(X_train, axis=1))
X_val_fft_mag = np.abs(np.fft.rfft(X_val, axis=1))
X_test_fft_mag = np.abs(np.fft.rfft(X_test, axis=1))

print("Training set shape:", X_train_fft_mag.shape)
print("Validation set shape:", X_val_fft_mag.shape)
print("Test set shape:", X_test_fft_mag.shape)

In [ ]:
# Training of the model with the same architecture as before.

mlp_fft_mag = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)
mlp_fft_mag.fit(X_train_fft_mag, y_train)

print("Epochs:", mlp_fft_mag.n_iter_)
print("Final training loss:", mlp_fft_mag.loss_)

In [ ]:
# Results

from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = mlp_fft_mag.predict(X_val_fft_mag)

print("Validation results:")
print(classification_report(
    y_val,
    y_val_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val, y_val_pred, labels=[0, 1, 2, 3, 4]
))

FFT magnitude reduces the input from 250 to 126 features but
discards phase information. Its validation macro F1 is 0.9113,
compared with 0.9373 for real and imaginary FFT features.
The best tested MLP remains the raw-data network with two
hidden layers of 100 neurons, achieving 0.9375.

## NN for 15-class classification

Lastly, we want to try the 15-class classification using the best-performing architecture.

First, let's recover the original lables.

In [ ]:
y_train_15 = train_data.iloc[:, -1].astype(int).to_numpy()
y_val_15 = val_data.iloc[:, -1].astype(int).to_numpy()
y_test_15 = test_data.iloc[:, -1].astype(int).to_numpy()

Now we can train the model as we did for all the other cases.

In [ ]:
mlp_raw_15 = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    max_iter=200,
    shuffle=True,
    random_state=42,
    verbose=True,
)

mlp_raw_15.fit(X_train, y_train_15)

print("Epochs:", mlp_raw_15.n_iter_)
print("Final training loss:", mlp_raw_15.loss_)

Results:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

y_val_pred_15 = mlp_raw_15.predict(X_val)
labels_15 = list(range(15))

print("Validation results:")
print(classification_report(
    y_val_15,
    y_val_pred_15,
    labels=labels_15,
    digits=3,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_val_15,
    y_val_pred_15,
    labels=labels_15,
))

print("Macro F1 on classes present in validation:",
      f1_score(
          y_val_15,
          y_val_pred_15,
          labels=np.unique(y_val_15),
          average="macro",
          zero_division=0,
      ))

Notice that some class 9 has only 2 training samples and none for validation and test, so we can't evaluate generalization performance.

## Conclusion

In order for us to conclude the model design using neural networks, let's see the performance of the best performing model on the test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred = mlp_raw.predict(X_test)

print("Test results:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["N", "S", "V", "F", "Q"],
    digits=4,
    zero_division=0,
))

print("Confusion matrix (rows: true classes, columns: predictions):")
print(confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1, 2, 3, 4],
))